# 🧪 W2-D2 概念实验：Tokenizer 与词嵌入

> 配套阅读：`第2周-Day2-Tokenizer与词嵌入.md`（完整原理、SentencePiece、业务关联在那边）
> 这个 notebook 用可执行实验回答三个问题：
> 1. **BPE 分词到底是怎么学的？** 从字符开始合并的过程
> 2. **词嵌入的相似度意味着什么？** 余弦相似度与"类比推理"
> 3. **词表大小 vs token 数量的权衡**

## 实验 1：BPE 合并过程 — 从字符到子词

BPE 核心思想：从字符开始，不断合并语料中出现频率最高的相邻字符对，直到词表达到目标大小。

In [ ]:
from collections import Counter

corpus = "low lower newest widest low low newest lower wide"
word_freqs = Counter(corpus.split())

# 初始化：每个词拆成字符
splits = {word: list(word) for word in word_freqs}
merges_history = []

def get_pair_freqs(splits, word_freqs):
    pair_freqs = Counter()
    for word, freq in word_freqs.items():
        s = splits[word]
        for i in range(len(s) - 1):
            pair_freqs[(s[i], s[i+1])] += freq
    return pair_freqs

def merge_pair(splits, pair):
    for word in splits:
        s = splits[word]
        i = 0
        while i < len(s) - 1:
            if (s[i], s[i+1]) == pair:
                s[i] = s[i] + s[i+1]
                del s[i+1]
            else:
                i += 1

print("=== BPE 训练过程 ===")
print(f"初始词: {dict(splits)}\n")

for step in range(12):
    pair_freqs = get_pair_freqs(splits, word_freqs)
    if not pair_freqs:
        break
    best = pair_freqs.most_common(1)[0][0]
    merges_history.append(best)
    merge_pair(splits, best)
    vocab_size = len(set(c for s in splits.values() for c in s))
    print(f"步骤 {step+1}: 合并 '{best[0]}'+'{best[1]}' → '{best[0]+best[1]}' (频次={pair_freqs[best]})  词表大小={vocab_size}")

print(f"\n最终分词结果:")
for word in sorted(splits):
    print(f"  {word} → {splits[word]}")

## 实验 2：词嵌入的余弦相似度 — 向量空间中的"语义距离"

用手工构造的小词表，验证：
- 语义相近的词向量余弦相似度高
- 类比推理：king - man + woman ≈ queen

In [ ]:
import numpy as np

# 手工构造 4D 嵌入（模拟语义关系）
# 维度语义：[王室, 性别, 年龄, 国籍]
vocab = {
    '国王': np.array([ 1.0,  0.8, 0.5, 0.3]),
    '女王': np.array([ 1.0, -0.8, 0.5, 0.3]),
    '男人': np.array([ 0.1,  0.8, 0.3, 0.5]),
    '女人': np.array([ 0.1, -0.8, 0.3, 0.5]),
    '王子': np.array([ 0.9,  0.7, 0.1, 0.3]),
    '公主': np.array([ 0.9, -0.7, 0.1, 0.3]),
}

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# 相似度矩阵
words = list(vocab.keys())
sim_matrix = np.zeros((len(words), len(words)))
for i, w1 in enumerate(words):
    for j, w2 in enumerate(words):
        sim_matrix[i, j] = cosine_sim(vocab[w1], vocab[w2])

print("余弦相似度矩阵:")
print(f"{'':>6}" + ''.join(f'{w:>6}' for w in words))
for i, w1 in enumerate(words):
    print(f"{w1:>6}" + ''.join(f'{sim_matrix[i,j]:6.3f}' for j in range(len(words))))

# 类比推理: 国王 - 男人 + 女人 ≈ ?
result = vocab['国王'] - vocab['男人'] + vocab['女人']
print("\n类比推理: 国王 - 男人 + 女人 =")
best_word = max(words, key=lambda w: cosine_sim(result, vocab[w]))
for w in sorted(words, key=lambda w: cosine_sim(result, vocab[w]), reverse=True):
    print(f"  {w}: {cosine_sim(result, vocab[w]):.4f}")
print(f"\n最接近的是「{best_word}」— 类比推理成功！")

## 实验 3：词表大小 vs Token 数量 — "大词表省 token" 的代价

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

# 模拟：处理固定语料，不同词表大小时的平均 token 数
# 词表越大，高频组合被合并 → token 数减少；但 embedding 参数线性增长
vocab_sizes = [100, 500, 2000, 5000, 10000, 30000, 65000, 100000]
# 词表越大 token 数越少（边际递减）
tokens_per_1k_chars = [950, 800, 650, 520, 440, 380, 340, 310]
# embedding 参数量 = vocab_size × d_model (d_model=2048)
d_model = 2048
embed_params_M = [v * d_model / 1e6 for v in vocab_sizes]

fig, ax1 = plt.subplots(figsize=(8, 4))
color1 = 'tab:blue'
ax1.plot(vocab_sizes, tokens_per_1k_chars, 'o-', color=color1)
ax1.set_xlabel('词表大小')
ax1.set_ylabel('每千字 token 数', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)

ax2 = ax1.twinx()
color2 = 'tab:red'
ax2.plot(vocab_sizes, embed_params_M, 's--', color=color2)
ax2.set_ylabel('Embedding 参数量 (M)', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('词表大小的两难：token 效率 ↑ vs 参数量 ↑')
plt.tight_layout()
plt.show()

print("读图：")
print("- 左轴（蓝）：词表越大，每千字所需 token 越少（推理更快）")
print("- 右轴（红）：Embedding 层参数量线性增长（内存更大）")
print("- Qwen2 (100K+) 在中文场景下比 LLaMA (32K) 每条 prompt 省 2-3x token")

## 实验 4：BPE 的 OOV 处理 — 未知词自动拆分为子词

In [ ]:
# 演示 BPE 遇到 OOV 时的行为：从未见过的词也能分解
# 假设训练词表中有 low/lower/new/newest/widest/wide 的合并规则
trained_merges = [('o','w'),('e','s'),('n','e'),('w','e'),('ne','w'),('lo','w'),('l','o'),('i','d'),('wi','d'),('ne','st')]

def tokenize_bpe(word, merges):
    tokens = list(word)
    for pair in merges:
        merged = pair[0] + pair[1]
        i = 0
        while i < len(tokens) - 1:
            if tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                tokens[i] = merged
                del tokens[i+1]
            else:
                i += 1
    return tokens

test_words = ["lowest", "newer", "widely", "unknown", "abclowdef"]
print("BPE 对未见词的分词结果（所有字符都在基础词表中，一定能拆分）:")
for w in test_words:
    tokens = tokenize_bpe(w, trained_merges)
    print(f"  {w} → {tokens}")

print("\n结论：BPE 永远不会产生真正的 OOV——未知词退化为更细的子词/字符序列。")
print("这是 BPE 相比词级分词的核心优势。")